In [1]:
import random
import pandas as pd

In [2]:
# Define intents and patient message templates 
intents = {
    "discuss_symptoms": [
        "I've had a persistent headache for three days now with blurred vision.",
        "There's sharp pain in my lower right abdomen that comes and goes.",
        "I'm experiencing shortness of breath even with minimal activity.",
        "I have severe diarrhea and vomiting since last night - can't keep anything down.",
        "My joints are swollen and painful, especially in the mornings.",
        "I noticed blood in my urine this morning with some burning sensation.",
        "I've been having chest palpitations that last several minutes at a time.",
        "There's a strange numbness in my left arm that won't go away."
    ],
    "request_appointment": [
        "I need to schedule a physical exam before my overseas trip next month.",
        "Can I book a follow-up with Dr. Smith for my post-op check?",
        "I'd like to make a same-day urgent appointment if possible.",
        "Are there any weekend slots available for a general consultation?",
        "I need to schedule my annual diabetes screening and checkup.",
        "Could I get an appointment with the cardiology department?",
        "I'd prefer a telehealth visit if available - when's the next opening?",
        "My child needs a sports physical - do you have pediatric appointments tomorrow?"
    ],
    "ask_prescription": [
        "My insulin prescription is running low - can you send a refill to CVS?",
        "I'm traveling abroad and need a 3-month supply of my blood thinners.",
        "The antidepressant dosage doesn't seem effective anymore - can we adjust?",
        "I lost my prescription slip for the antibiotics - can you reissue?",
        "My pharmacy says they need prior authorization for my new cholesterol meds.",
        "I'm having bad side effects from the new medication - can we switch?",
        "Do I need a new prescription for my ongoing thyroid medication?",
        "Can you prescribe something stronger for my chronic back pain?"
    ],
    "follow_up": [
        "It's been 2 weeks since my surgery - when should I come for a wound check?",
        "The antibiotics helped but my sinus infection isn't completely gone.",
        "My blood sugar levels have stabilized - do I still need the same dosage?",
        "The physical therapy exercises caused more pain - is this normal?",
        "I completed the treatment but still feel fatigued - what next?",
        "The biopsy results came back - can we discuss them at my next visit?",
        "I've been tracking my blood pressure as instructed - should I continue the meds?",
        "The rash has spread despite using the prescribed cream - what now?"
    ],
    "complaint": [
        "I've been waiting 45 minutes past my scheduled appointment time.",
        "The receptionist was very rude when I called to reschedule.",
        "The billing department charged me for services I never received.",
        "My test results were supposed to be ready yesterday but still aren't available.",
        "The doctor didn't explain my diagnosis clearly - I have many unanswered questions.",
        "The prescription you sent was out of stock at my pharmacy with no alternatives.",
        "Your online portal isn't working and I can't access my medical records.",
        "The nurse didn't return my urgent call about medication side effects."
    ],
    "general_questions": [
        "What are your clinic hours on holidays?",
        "Do you accept my new insurance plan?",
        "Where can I park when coming for my appointment?",
        "What COVID safety measures are you currently following?",
        "How do I get a copy of my medical records transferred to another provider?",
        "What's your policy on late cancellations?",
        "Which hospital are you affiliated with for emergencies?",
        "Do you provide interpreter services for non-English speakers?"
    ]
}

# Define creative doctor responses per intent
doctor_responses = {
    "discuss_symptoms": [
        "I understand this must be worrying. How long exactly have you been experiencing these symptoms?",
        "That does sound concerning. Have you noticed any triggers or patterns to when this occurs?",
        "Let me make sure I understand - is the pain constant or does it come in waves?",
        "Before we proceed, I need to ask: are you experiencing any fever along with these symptoms?",
        "Thank you for sharing these details. On a scale of 1-10, how would you rate your discomfort?",
        "I'd like to investigate this further. Have you had similar symptoms in the past?",
        "This requires attention. Are you currently taking any medications or have any known allergies?",
        "Let me check your records first. When was your last visit to the clinic?"
    ],
    "request_appointment": [
        "I can help with that. Would you prefer an early morning or late afternoon appointment?",
        "Let me check our availability. Are you looking for something this week or next?",
        "We have openings tomorrow - would that work for you?",
        "Dr. Smith has availability on Tuesday or Friday - which works better?",
        "For urgent cases, we keep same-day slots open. Can you come in within the next two hours?",
        "Our pediatric department has openings at 10am or 3pm tomorrow - which would you prefer?",
        "Telehealth appointments are available today after 2pm - would that suit you?",
        "I see we have a cancellation for this afternoon at 4pm - shall I reserve that for you?"
    ],
    "ask_prescription": [
        "I can certainly renew that. Has your current dosage been effective for you?",
        "Let me verify - you need a refill for [medication name], correct?",
        "For international travel, I can prescribe a 90-day supply with 3 refills.",
        "I'll need to check your last lab results before approving this refill.",
        "The system shows you're due for blood work before we renew this prescription.",
        "I can authorize that, but let's discuss the side effects you're experiencing first.",
        "This medication requires monitoring - when was your last checkup?",
        "I'll send that to your pharmacy, but recommend a follow-up in 2 weeks."
    ],
    "follow_up": [
        "Glad you're following up. How have your symptoms progressed since we last spoke?",
        "Let's review your case. What improvements have you noticed with the treatment?",
        "I have your test results here. There are few things we should discuss.",
        "The healing process can vary - are you still experiencing [specific symptom]?",
        "Based on your update, we may need to adjust your treatment plan.",
        "You mentioned new symptoms - when exactly did these begin?",
        "I'd like to schedule some additional tests based on your follow-up report.",
        "Your progress sounds encouraging. Let's discuss next steps."
    ],
    "complaint": [
        "I sincerely apologize for this experience - let me see how we can make it right.",
        "Thank you for bringing this to our attention. This isn't our usual standard.",
        "I'm documenting your concern and will personally follow up with the department involved.",
        "We take such feedback seriously. May I get more details to investigate properly?",
        "I understand your frustration. Here's what we can do to resolve this...",
        "You're right to expect better service. Let me connect you with our patient advocate.",
        "I'm sorry we fell short of your expectations. How can we make this right for you?",
        "This isn't acceptable. I'll escalate this to our practice manager immediately."
    ],
    "general_questions": [
        "Our regular hours are 8am-6pm weekdays, with urgent care until 8pm.",
        "We accept most major insurance plans. May I have your member ID to verify?",
        "There's free patient parking behind our building with validation at reception.",
        "We still require masks in clinical areas and have separate waiting zones.",
        "Records requests take 3-5 business days - I can email you the release form.",
        "We have a 24-hour cancellation policy to avoid a $50 no-show fee.",
        "Our physicians are affiliated with City General Hospital for emergencies.",
        "Yes, we offer professional interpreter services at no cost - which language?"
    ]
}

sentiments = ["positive", "neutral", "negative", "urgent", "formal", "friendly", "concerned", "apologetic"]

In [3]:
# Simple summary generator
def generate_summary(intent):
    summaries = {
        "discuss_symptoms": "Patient described current health symptoms.",
        "request_appointment": "Patient requested a future consultation.",
        "ask_prescription": "Patient asked for medication renewal or advice.",
        "follow_up": "Patient followed up regarding previous consultation.",
        "complaint": "Patient reported dissatisfaction or issues."
    }
    return summaries.get(intent, "General health chat.")

# Simple key point extractor
def extract_key_points(message):
    return ", ".join([word for word in message.split() if len(word) > 5][:3])


In [4]:
# Data generation function
def generate_data(num_samples=300):
    data = []

    for i in range(num_samples):
        intent = random.choice(list(intents.keys()))
        patient_msg = random.choice(intents[intent])
        doctor_reply = random.choice(doctor_responses[intent])
        sentiment = random.choice(sentiments)
        summary = generate_summary(intent)
        key_points = extract_key_points(patient_msg)

        data.append({
            "conversation_id": i + 1,
            "patient_message": patient_msg,
            "doctor_response": doctor_reply,
            "intent": intent,
            "sentiment": sentiment,
            "summary": summary,
            "key_points": key_points
        })

    return pd.DataFrame(data)

# Save to CSV
df = generate_data(500)
df.to_csv("realistic_synthetic_chat_data.csv", index=False)
print("✅ Realistic synthetic chat data saved to 'realistic_synthetic_chat_data.csv'")

✅ Realistic synthetic chat data saved to 'realistic_synthetic_chat_data.csv'
